In [9]:
import fitz

In [10]:
def extract_txt_from_pdf(pdf_path):
    # Open the PDF file
    document = fitz.open(pdf_path)
    text = ""
    
    # Iterate through each page
    for page_num in range(len(document)):
        page = document.load_page(page_num)
        text += page.get_text()
    
    return text

pdf='Gang Of Four.pdf'
extracted_text = extract_txt_from_pdf(pdf)
print(extracted_text[:500])  # Print the first 500 characters of the extracted text

 
 
 Design Pattern Framework™ 4.0 
 
 
Page 1 of 86 
 
 
Gang of Four 
 
 
Software Design Patterns 
 
 
 
 
 
 
 
Companion document to  
Design Pattern Framework™ 4.0 
 
 
by 
 
 
Data & Object Factory, LLC 
www.dofactory.com 
 
 
Copyright © 2008-2009, Data & Object Factory, LLC 
All rights reserved 
 
 
 Design Pattern Framework™ 4.0 
 
 
Copyright © 2006-2010, Data & Object Factory, LLC. All rights reserved.  
Page 2 of 86 
1. Index 
 
 
1. 
Index ..........................................


In [11]:
def split_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    text_length = len(text)
    
    while start < text_length:
        end = min(start + chunk_size, text_length)
        chunk = text[start:end]
        chunks.append(chunk)
        
        start += chunk_size - overlap  # Move the start index forward with overlap
    
    return chunks

text_chunks = split_text(extracted_text)
print(f"Total chunks created: {len(text_chunks)}")
print(text_chunks[0])  # Print the first chunk

Total chunks created: 302
 
 
 Design Pattern Framework™ 4.0 
 
 
Page 1 of 86 
 
 
Gang of Four 
 
 
Software Design Patterns 
 
 
 
 
 
 
 
Companion document to  
Design Pattern Framework™ 4.0 
 
 
by 
 
 
Data & Object Factory, LLC 
www.dofactory.com 
 
 
Copyright © 2008-2009, Data & Object Factory, LLC 
All rights reserved 
 
 
 Design Pattern Framework™ 4.0 
 
 
Copyright © 2006-2010, Data & Object Factory, LLC. All rights reserved.  
Page 2 of 86 
1. Index 
 
 
1. 
Index ..........................................


In [13]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(text_chunks)
print(f"Shape of embeddings: {embeddings.shape}")

ModuleNotFoundError: No module named 'sentence_transformers'

In [ ]:
import faiss
import numpy as np
embeddings_matrix = np.array(embeddings)
dimension = embeddings_matrix.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings_matrix)
print(f"Number of vectors in the index: {index.ntotal}")

In [ ]:
def search(query, top_k=3):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, top_k)
    results = [text_chunks[i] for i in indices[0]]
    return results

query = "What is the main topic of the book?"
k=3
results = search(query, k)

print("Search results:")
for i, result in enumerate(results):
    print(f"Result {i+1}:\n{result}\n")

In [ ]:
from openai import OpenAI
# in case you want to use the Groq API, you can specify the base_url as follows and add your api key:
client = OpenAI(base_url="http://api.groq.com/openai/v1", api_key="sk-xxx")

In [ ]:
def generate_response(query, context_chunks):
    context='\n\n'.join(context_chunks)
    prompt=f'You are a helpful assistant. Based on the following context, answer the question: "{query}"\n\nContext:\n{context}'
    response = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

In [ ]:
query = "what are the main design patterns in software engineering?"
top_chunks = search(query, top_k=3) # replace with k if error
answer = generate_response(query, top_chunks)
print(answer)